# GoEmotions project proposition

**Question.** On short Reddit comments, how do document representation and classifier depth affect multilabel emotion recognition?

GoEmotions contains 43,410 train, 5,426 development, and 5,427 test comments with 27 emotions plus `neutral`; 16.36% of training examples have multiple labels. Training support ranges from 77 (`grief`) to 14,219 (`neutral`). Therefore this is a strongly imbalanced **multilabel**, not multiclass, task. The primary metric is macro-F1 (every emotion has equal weight); micro-F1 is reported as context. The official test set stays untouched until the final evaluation.

This notebook covers the assignment checklist with a full $3\times3$ comparison:

| Requirement | Choices |
|---|---|
| Representations | BPE token indicators; BPE token-count bag of words; BPE TF-IDF |
| Architectures | one-vs-rest log-linear; one-hidden-layer MLP; two-hidden-layer MLP |
| Comparison | identical shuffled folds, macro/micro-F1, mean and fold variation |
| Bayesian analysis | correlated $t$ posterior for the best two systems; ROPE $=\pm0.005$ macro-F1 |

**Tutorial alignment.** As in `tutorial.ipynb`, a Hugging Face BPE tokenizer uses Metaspace pre-tokenization, `vocab_size=1000`, and `min_frequency=2`; it is trained once on the official training texts and saved for reproducibility. Indicators and counts are the tutorial encoders; TF-IDF is the required extra representation. The tutorial's multiclass softmax models cannot represent multiple simultaneous emotions, so their GoEmotions adaptations use 28 sigmoid decisions and binary cross-entropy while retaining the same linear/hidden-layer distinction.

In [ ]:
# Cell 1 — Configure reproducibility, locate the archive, and load all official splits.
from pathlib import Path
import csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baycomp
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.trainers import BpeTrainer
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.model_selection import KFold, cross_validate
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer

SEED = 0
CV_SEEDS = [0]            # Tutorial default; use [0, 3, 5] for repeated CV if time permits
QUICK_RUN = False         # True = 6,000-row smoke test; False = reportable full experiment
N_SPLITS = 3 if QUICK_RUN else 5
MAX_ROWS = 6_000 if QUICK_RUN else None
BPE_VOCAB_SIZE = 1_000
BPE_MIN_FREQUENCY = 2
ROPE = 0.005              # tutorial convention: half a macro-F1 percentage point

def locate_data():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'archive' / 'data'
        if (candidate / 'train.tsv').exists():
            return candidate
    raise FileNotFoundError('Could not find archive/data from the current working directory.')

DATA_DIR = locate_data()
LABELS = (DATA_DIR / 'emotions.txt').read_text(encoding='utf-8').splitlines()
mlb = MultiLabelBinarizer(classes=np.arange(len(LABELS))).fit([[]])

def read_split(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.tsv', sep='\t', header=None,
                        names=['text', 'label_ids', 'id'], quoting=csv.QUOTE_NONE)
    frame['label_ids'] = frame['label_ids'].map(lambda s: [int(i) for i in s.split(',')])
    return frame, mlb.transform(frame['label_ids'])

splits = {name: read_split(name) for name in ('train', 'dev', 'test')}
train, Y_train = splits['train']
rng = np.random.default_rng(SEED)
cv_idx = (np.sort(rng.choice(len(train), MAX_ROWS, replace=False))
          if MAX_ROWS and MAX_ROWS < len(train) else np.arange(len(train)))
X_cv, Y_cv = train.iloc[cv_idx]['text'].reset_index(drop=True), Y_train[cv_idx]

# Tutorial recipe: train BPE on training text only and persist it.
RESULTS_DIR = DATA_DIR.parents[1] / 'goemotions_results'
RESULTS_DIR.mkdir(exist_ok=True)
BPE_PATH = RESULTS_DIR / f'bpe_vs{BPE_VOCAB_SIZE}_mf{BPE_MIN_FREQUENCY}.json'
tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = Metaspace()
trainer = BpeTrainer(vocab_size=BPE_VOCAB_SIZE, min_frequency=BPE_MIN_FREQUENCY,
                     special_tokens=['[UNK]'])
tokenizer.train_from_iterator(train['text'], trainer=trainer)
tokenizer.save(str(BPE_PATH))
BPE_VOCAB = tokenizer.get_vocab()

def bpe_analyzer(text):
    return tokenizer.encode(text).tokens

print(f'Data: {DATA_DIR} | mode: {"QUICK" if QUICK_RUN else "FULL"} | CV rows: {len(X_cv):,}')
print(f'BPE vocabulary: {len(BPE_VOCAB):,} tokens | saved to: {BPE_PATH}')

In [ ]:
# Cell 2 — Verify split sizes, label cardinality, imbalance, and multilabel frequency.
overview = []
for name, (frame, Y) in splits.items():
    per_item = Y.sum(axis=1)
    overview.append({'split': name, 'examples': len(frame),
                     'mean_labels': per_item.mean(), 'multilabel_%': 100 * (per_item > 1).mean()})
display(pd.DataFrame(overview).round(2))

label_counts = pd.DataFrame({'label': LABELS, 'train_count': Y_train.sum(axis=0)})
display(label_counts.sort_values('train_count', ascending=False).reset_index(drop=True))
ax = label_counts.sort_values('train_count').plot.barh(
    x='label', y='train_count', figsize=(7, 8), legend=False, title='Training-label imbalance')
ax.set(xlabel='positive examples', ylabel=''); plt.tight_layout()
sample = train.iloc[0]['text']
print('Sample text:', sample)
print('BPE tokens:', bpe_analyzer(sample))

## Experiment design

All systems use the same saved BPE vocabulary, exactly separating **tokenization** from **document encoding** as the tutorial does. Binary indicators test BPE-token presence, counts implement the tutorial bag of words, and sublinear TF-IDF is the additional representation. IDF values and every classifier are estimated only on each fold's training partition.

The log-linear model uses one logistic decision per label. The one-hidden-layer MLP follows the tutorial's 128-unit ReLU design, batch size 64, learning rate $10^{-4}$, weight decay $10^{-4}$, and 10 epochs. Scikit-learn's multilabel MLP uses sigmoid outputs and binary cross-entropy; its `alpha` is the tutorial's weight decay, but it does not expose dropout. A two-hidden-layer `(128, 64)` MLP is the required architectural extension.

The tutorial's shuffled five-fold procedure is reproduced with identical paired folds for every system. Random K-fold is not iterative multilabel stratification, but even the rarest training label has 77 positives, making an empty full-data fold unlikely. The official test set remains untouched until selection is complete.

In [ ]:
# Cell 3 — Construct exactly three representations × three architectures (nine pipelines).
bpe_args = dict(analyzer=bpe_analyzer, vocabulary=BPE_VOCAB, lowercase=False)
representations = {
    'BPE indicator': CountVectorizer(binary=True, **bpe_args),
    'BPE count': CountVectorizer(binary=False, **bpe_args),
    'BPE tfidf': TfidfVectorizer(sublinear_tf=True, norm='l2', **bpe_args),
}
architectures = {
    'log-linear': OneVsRestClassifier(
        LogisticRegression(solver='liblinear', max_iter=100, random_state=SEED), n_jobs=1),
    'MLP-1': MLPClassifier(hidden_layer_sizes=(128,), activation='relu', solver='adam',
                           batch_size=64, learning_rate_init=1e-4, alpha=1e-4,
                           max_iter=10, early_stopping=False, random_state=SEED),
    'MLP-2': MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                           batch_size=64, learning_rate_init=1e-4, alpha=1e-4,
                           max_iter=10, early_stopping=False, random_state=SEED),
}
systems = {f'{rep} + {arch}': make_pipeline(clone(vectorizer), clone(model))
           for rep, vectorizer in representations.items()
           for arch, model in architectures.items()}
list(systems)

In [ ]:
# Cell 4 — Run paired cross-validation and rank systems by the primary metric, macro-F1.
scoring = {
    'macro_f1': make_scorer(f1_score, average='macro', zero_division=0),
    'micro_f1': make_scorer(f1_score, average='micro', zero_division=0),
}
rows = []
for system, pipeline in systems.items():
    start = time.time()
    for run, seed in enumerate(CV_SEEDS):
        cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        result = cross_validate(pipeline, X_cv, Y_cv, cv=cv, scoring=scoring,
                                n_jobs=1, error_score='raise')
        for fold in range(N_SPLITS):
            rows.append({'system': system, 'run': run, 'seed': seed, 'fold': fold,
                         'macro_f1': result['test_macro_f1'][fold],
                         'micro_f1': result['test_micro_f1'][fold],
                         'fit_seconds': result['fit_time'][fold]})
    print(f'{system:27s} finished in {(time.time() - start) / 60:.1f} min')

scores = pd.DataFrame(rows)
summary = (scores.groupby('system', as_index=False)
           .agg(macro_f1=('macro_f1', 'mean'), macro_sd=('macro_f1', 'std'),
                micro_f1=('micro_f1', 'mean'), mean_fit_s=('fit_seconds', 'mean'))
           .sort_values('macro_f1', ascending=False).reset_index(drop=True))
display(summary.round(4))
scores.boxplot(column='macro_f1', by='system', rot=45, figsize=(10, 4));
plt.suptitle(''); plt.title('Paired fold-level macro-F1'); plt.tight_layout()

## Bayesian practical comparison

The top two systems are compared on their paired fold scores with the tutorial's Bayesian correlated $t$ model. Its posterior accounts for overlap among cross-validation training sets. Following the tutorial, the ROPE $[-0.005, 0.005]$ says that an expected difference below half a macro-F1 percentage point is practically negligible. Report the probabilities of A winning, practical equivalence, and B winning rather than a binary significance claim.

In [ ]:
# Cell 5 — Estimate and visualise the posterior difference between the best two systems.
A, B = summary.loc[0, 'system'], summary.loc[1, 'system']
order = ['run', 'fold']
a = scores.loc[scores.system.eq(A)].sort_values(order)['macro_f1'].to_numpy()
b = scores.loc[scores.system.eq(B)].sort_values(order)['macro_f1'].to_numpy()
posterior = baycomp.CorrelatedTTest(a, b, rope=ROPE, runs=len(CV_SEEDS))  # B − A
p_A, p_rope, p_B = posterior.probs()
print(f'A: {A}\nB: {B}')
print(f'P(A wins)={p_A:.3f} | P(ROPE)={p_rope:.3f} | P(B wins)={p_B:.3f}')
print(f'posterior mean(B−A)={posterior.mean:.4f}; effect size={posterior.mean / np.sqrt(posterior.var):.2f}')
samples = np.asarray(posterior.sample)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(samples, bins=80, density=True, alpha=.75)
ax.axvspan(-ROPE, ROPE, color='teal', alpha=.2, label=f'ROPE ±{ROPE}')
ax.axvline(0, color='black', lw=1); ax.set(xlabel='posterior macro-F1 difference (B − A)', ylabel='density')
ax.legend(); plt.tight_layout()

In [ ]:
# Cell 6 — Refit the selected system on train+dev, evaluate test once, and expose errors.
best_name = A
dev, Y_dev = splits['dev']; test, Y_test = splits['test']
X_fit = pd.concat([train['text'], dev['text']], ignore_index=True)
Y_fit = np.vstack([Y_train, Y_dev])
if QUICK_RUN and MAX_ROWS is not None and MAX_ROWS < len(X_fit):
    final_idx = np.sort(rng.choice(len(X_fit), MAX_ROWS, replace=False))
    X_fit, Y_fit = X_fit.iloc[final_idx], Y_fit[final_idx] 

final_model = clone(systems[best_name]).fit(X_fit, Y_fit)
Y_pred = final_model.predict(test['text'])
final_metrics = pd.Series({
    'macro_f1': f1_score(Y_test, Y_pred, average='macro', zero_division=0),
    'micro_f1': f1_score(Y_test, Y_pred, average='micro', zero_division=0),
    'subset_accuracy': accuracy_score(Y_test, Y_pred),
}, name=best_name)
display(final_metrics.to_frame())

per_label = pd.DataFrame({'label': LABELS, 'support': Y_test.sum(axis=0),
                          'predicted': Y_pred.sum(axis=0),
                          'f1': f1_score(Y_test, Y_pred, average=None, zero_division=0)})
display(per_label.sort_values('f1').reset_index(drop=True))

def decode(row):
    return ', '.join(np.asarray(LABELS)[np.flatnonzero(row)]) or '∅'
mistakes = np.flatnonzero((Y_test != Y_pred).any(axis=1))[:10]
error_examples = pd.DataFrame({
    'text': test.iloc[mistakes]['text'].to_numpy(),
    'gold': [decode(Y_test[i]) for i in mistakes],
    'predicted': [decode(Y_pred[i]) for i in mistakes],
})
display(error_examples)

## How to finish and report

1. Use `QUICK_RUN=True` only to verify the notebook. For reportable numbers set it to `False`, restart the kernel, and run all cells.
2. Explain the tutorial transfer: BPE segmentation → indicators/counts → log-linear/MLP classifiers → paired cross-validation → Bayesian estimation. Explicitly justify sigmoid/BCE instead of softmax/NLL because GoEmotions is multilabel.
3. Report the ranked table, fold variation, the three posterior probabilities, and final test macro/micro-F1. Explain whether the runner-up is practically equivalent before preferring a more complex model.
4. Inspect the per-label table and error examples. Discuss rare-label recall, confusable emotions, multilabel misses, and whether TF-IDF or added depth helps. Subset accuracy is harsh because every label must match.
5. Limitations: the BPE tokenizer follows the tutorial by using all official training texts before CV (an unsupervised, transductive preprocessing choice); Reddit is not globally representative; labels are subjective; rare emotions make macro-F1 uncertain; random K-fold is not iterative multilabel stratification.

**Sources.** Demszky et al. (2020), *GoEmotions: A Dataset of Fine-Grained Emotions*; Benavoli et al. (2017), *Time for a Change: a Tutorial for Comparing Multiple Classifiers Through Bayesian Analysis*.

**AI-assistance record.** An AI coding assistant helped inspect the supplied files and draft this experimental notebook. The student remains responsible for running it, checking the results, understanding every design choice, and documenting any later AI-assisted changes.